# DEQ Export Demo — `lightstim.deq`

**[DEMO]** — the exporter is packaged in `lightstim/deq/`; this notebook only imports and demonstrates it.

Converts LightStim circuits into Microsoft's `.deq` DSL (from the open-source
[`qdk-ec`](https://github.com/microsoft/qdk-ec) toolkit). See the Light-DEQ plan for
background: this exporter replaces a from-scratch `.deq` generator that used to live in
`resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq`, whose
generator was deleted but whose output (`generated/rotated_surface_code_d3.deq`) was kept
as a reference target.

This notebook:
1. Exports a small repetition-code memory circuit and checks the `.deq` text structurally.
2. Exports LightStim's own rotated surface code (d=3) and compares it against the
   resource-superstaq reference file's `CODE` parameters.
3. Notes the current scope/limitations.


In [1]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.repetition.repetition import RepetitionCode
from lightstim.qec_code.surface_code.rotated import (
    RotatedSurfaceCode,
    RotatedSurfaceCodeExtractionBlock,
)
from lightstim.deq import export_deq
from lightstim.deq.validate import deq_available, validate_deq_text

print("deq/deqagram grammar available for validation:", deq_available())


deq/deqagram grammar available for validation: True


## 1. Repetition code (d=3) — exact structural check

The exported circuit must be **noiseless**: pass `noise_params=None` to `MemoryExperiment`.

In [2]:
rep_patch = RepetitionCode(distance=3)
rep_exp = MemoryExperiment(qec_patch=rep_patch, rounds=3, noise_params=None, basis="Z")
rep_circuit = rep_exp.build()

print(f"Qubits: {rep_circuit.num_qubits}  Detectors: {rep_circuit.num_detectors}  "
      f"Observables: {rep_circuit.num_observables}")

rep_deq_text = export_deq(rep_exp.system, rep_circuit, gadget_name="Repetition")
print(rep_deq_text)


Qubits: 5  Detectors: 8  Observables: 1
# Generated by lightstim.deq.export; do not edit by hand.

CODE memory [[3,1,3]] {
    LOGICAL X0*X1*X2 Z0
    STABILIZER Z0*Z1
    STABILIZER Z1*Z2
}

GADGET Repetition {
    R 0 2 4 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    TICK
    CX 2 1 4 3
    TICK
    TICK
    M 1 3
    DETECTOR rec[-2]
    DETECTOR rec[-1]
    TICK
    R 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    TICK
    CX 2 1 4 3
    TICK
    TICK
    M 1 3
    DETECTOR rec[-4] rec[-2]
    DETECTOR rec[-3] rec[-1]
    REPEAT 1 {
        TICK
        R 1 3
        TICK[SE_start]
        TICK
        CX 0 1 2 3
        TICK
        CX 2 1 4 3
        TICK
        TICK
        M 1 3
        DETECTOR rec[-4] rec[-2]
        DETECTOR rec[-3] rec[-1]
    }
    TICK
    M 0 2 4
    DETECTOR rec[-5] rec[-3] rec[-2]
    DETECTOR rec[-4] rec[-2] rec[-1]
    OBSERVABLE_INCLUDE rec[-3]
    OUTPUT memory 0 2 4
}



In [3]:
rep_parsed = validate_deq_text(rep_deq_text)
print("Parsed OK:", rep_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition, GadgetDefinition
    code_def = next(d for d in rep_parsed.definitions if isinstance(d, CodeDefinition))
    gadget_def = next(d for d in rep_parsed.definitions if isinstance(d, GadgetDefinition))
    print(f"CODE {code_def.name}: n={code_def.n} k={code_def.k} d={code_def.d}, "
          f"{len(code_def.logicals)} logical(s), {len(code_def.stabilizers)} stabilizer(s)")
    print(f"GADGET {gadget_def.name}: {len(gadget_def.body)} body statements")
    assert (code_def.n, code_def.k, code_def.d) == (3, 1, 3)
    assert len(code_def.stabilizers) == 2  # d-1 Z-checks


Parsed OK: True
CODE memory: n=3 k=1 d=3, 1 logical(s), 2 stabilizer(s)
GADGET Repetition: 30 body statements


## 2. Rotated surface code (d=3) — primary Light-DEQ target

Structural comparison against
`resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq/generated/rotated_surface_code_d3.deq`,
whose `CODE RotatedSurfaceCodeW3H3 [[9,1,3]]` block has 8 stabilizers and 1 logical pair.
Exact qubit-index equality isn't expected (different indexing schemes between LightStim's
IR and that project's from-scratch geometry code) — only the algebraic shape should match.


In [4]:
rsc_patch = RotatedSurfaceCode(distance=3)
rsc_exp = MemoryExperiment(
    qec_patch=rsc_patch,
    extraction_block_class=RotatedSurfaceCodeExtractionBlock,
    rounds=3,
    noise_params=None,
    basis="Z",
)
rsc_circuit = rsc_exp.build()

print(f"Qubits: {rsc_circuit.num_qubits}  Detectors: {rsc_circuit.num_detectors}  "
      f"Observables: {rsc_circuit.num_observables}")

rsc_deq_text = export_deq(rsc_exp.system, rsc_circuit, gadget_name="RotatedSurfaceMemory")
print(rsc_deq_text[:1500] + "\n... (truncated) ...")


Qubits: 17  Detectors: 24  Observables: 1
# Generated by lightstim.deq.export; do not edit by hand.

CODE memory [[9,1,3]] {
    LOGICAL X0*X3*X6 Z0*Z1*Z2
    STABILIZER X0*X1
    STABILIZER Z0*Z1*Z3*Z4
    STABILIZER Z2*Z5
    STABILIZER X1*X2*X4*X5
    STABILIZER Z3*Z6
    STABILIZER Z4*Z5*Z7*Z8
    STABILIZER X3*X4*X6*X7
    STABILIZER X7*X8
}

GADGET RotatedSurfaceMemory {
    R 1 2 3 7 8 9 13 14 15 0 4 5 6 10 11 12 16
    TICK[SE_start]
    H 0 6 12 16
    TICK
    CX 0 2 6 9 12 14 8 4 13 10 15 11
    TICK
    CX 0 1 6 8 12 13 2 4 7 10 9 11
    TICK
    CX 6 3 12 8 16 15 7 4 9 5 14 11
    TICK
    CX 6 2 12 7 16 14 1 4 3 5 8 11
    TICK
    H 0 6 12 16
    TICK
    M 0 4 5 6 10 11 12 16
    DETECTOR rec[-7]
    DETECTOR rec[-6]
    DETECTOR rec[-4]
    DETECTOR rec[-3]
    TICK
    R 0 4 5 6 10 11 12 16
    TICK[SE_start]
    H 0 6 12 16
    TICK
    CX 0 2 6 9 12 14 8 4 13 10 15 11
    TICK
    CX 0 1 6 8 12 13 2 4 7 10 9 11
    TICK
    CX 6 3 12 8 16 15 7 4 9 5 14 11
    TICK
 

In [5]:
rsc_parsed = validate_deq_text(rsc_deq_text)
print("Parsed OK:", rsc_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition
    rsc_code_def = next(d for d in rsc_parsed.definitions if isinstance(d, CodeDefinition))
    print(f"CODE {rsc_code_def.name}: n={rsc_code_def.n} k={rsc_code_def.k} d={rsc_code_def.d}, "
          f"{len(rsc_code_def.logicals)} logical(s), {len(rsc_code_def.stabilizers)} stabilizer(s)")
    assert (rsc_code_def.n, rsc_code_def.k, rsc_code_def.d) == (9, 1, 3)
    assert len(rsc_code_def.stabilizers) == 8
    print("Matches resource-superstaq reference CODE RotatedSurfaceCodeW3H3 [[9,1,3]] "
          "(8 stabilizers, 1 logical pair).")


Parsed OK: True
CODE memory: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)
Matches resource-superstaq reference CODE RotatedSurfaceCodeW3H3 [[9,1,3]] (8 stabilizers, 1 logical pair).


In [6]:
# Optional: parse the actual resource-superstaq reference file directly, if present
# on this machine, and print its CODE parameters side by side for comparison.
from deq.circuit import parser as _deq_parser
from deq.circuit.model import CodeDefinition as _CodeDefinition

ref_path = (
    ROOT.parent
    / "resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq"
    / "generated/rotated_surface_code_d3.deq"
)
if deq_available() and ref_path.exists():
    ref_parsed = _deq_parser.parse(ref_path.read_text())
    ref_code_def = next(
        d for d in ref_parsed.definitions
        if isinstance(d, _CodeDefinition) and d.name == "RotatedSurfaceCodeW3H3"
    )
    print(f"reference: n={ref_code_def.n} k={ref_code_def.k} d={ref_code_def.d}, "
          f"{len(ref_code_def.logicals)} logical(s), {len(ref_code_def.stabilizers)} stabilizer(s)")
    print(f"lightstim: n={rsc_code_def.n} k={rsc_code_def.k} d={rsc_code_def.d}, "
          f"{len(rsc_code_def.logicals)} logical(s), {len(rsc_code_def.stabilizers)} stabilizer(s)")
else:
    print("Reference file or deq/deqagram not available in this environment; skipping direct comparison.")


reference: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)
lightstim: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)


## 3. Scope & limitations (v1)

- **Single-patch systems only.** `export_deq` raises `DeqExportError` for multi-patch
  `QECSystem`s (lattice surgery, code deformation) — that decomposition needs deq's
  `IN<p>.S<s>` virtual-port syntax and is tracked as follow-up.
- **Noiseless circuits only.** DEQ's `ERROR(p) <target>` statement takes CHECK/READOUT/LOGICAL
  targets, not per-qubit Pauli targets, so it isn't a drop-in for Stim's `X_ERROR(p) q...`.
  Pass `noise_params=None` to `MemoryExperiment` (or otherwise use the pre-noise-injection
  circuit) before exporting.
- **One monolithic GADGET per circuit**, not split into Prepare/SyndromeExtraction/Measure
  sub-gadgets wired via `COMPOSE` (unlike the resource-superstaq reference file). Splitting
  naively breaks `rec[-k]` scoping, since deq scopes measurement records *per GADGET* and a
  DETECTOR comparing against a prior gadget's measurement would go out of range.
- **PPVM-backed simulation is not yet implemented** — blocked on a Rust toolchain install
  in this environment; see the Light-DEQ plan for the intended design
  (`lightstim/simulation/ppvm_backend/`).
